# History Length Effect on XGBoost Prediction — M29 D23

Tests how the **XGBoost history window** affects prediction quality and rate map
reconstruction quality for target cell **214** (grid cell), using **all other grid cells**
as covariates alongside position.

History lengths tested: **50 ms, 100 ms, 500 ms, 1000 ms**

Two sessions: **VR** (1D position, trial structure) and **OF1** (2D open field).

**Figure layout:** 2 rows (VR top, OF1 bottom) × (True + n_histories) columns.
Each panel shows the reconstructed rate map; pR² is annotated in the title.

In [ ]:
import numpy as np
import pandas as pd
import pynapple as nap
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import LinearSegmentedColormap
from scipy.ndimage import gaussian_filter
from spatial_manifolds.detect_grids import *
from spatial_manifolds.mlencoding import *

import warnings
warnings.filterwarnings('ignore')
%load_ext autoreload
%autoreload 2
%matplotlib inline

plt.rcParams['font.family'] = 'Arial'

mouse       = 29
day         = 23
source_path = '/Users/harryclark/Downloads/COHORT12/'
fig_path    = '/Users/harryclark/Documents/spatial-manifolds/scripts/figures/figure_playground/'

TARGET_ID      = 214
# Minimum history = 3×time_bs = 30 ms. The raised cosine filter needs
# n_bins ≥ 3 so that nt[1] ≠ nt[-1] (otherwise db=0 → arange fails).
HISTORY_LENGTHS  = [50, 100, 500, 1000]  # ms
FIXED_NFILTERS   = 5   # fixed across all history lengths for fair comparison
                       # (proportional scaling confounds window length with model complexity)
N_CV           = 5
SIGMA          = 2.5   # gaussian smoothing for rate maps

COL_GC = '#c04744'

def white_to_hex_cmap(hex_color):
    return LinearSegmentedColormap.from_list('cmap', ['#FFFFFF', hex_color])

cell_class = pd.read_csv('/Users/harryclark/Documents/spatial-manifolds/data/cell_classifications.csv')
sess_cells = cell_class[
    (cell_class['mouse'] == mouse) & (cell_class['day'] == day)
].copy()
sess_cells['cluster_id'] = sess_cells['cluster_id'].astype(int)
print(f'Target cell: {TARGET_ID}')
print(f'History lengths: {HISTORY_LENGTHS} ms')

## 1. Load session data

In [ ]:
print('Loading VR...')
tcs_vr, tcs_time_vr, _, last_ephys_bin_vr, beh_vr, clusters_vr = compute_vr_tcs(
    mouse, day, apply_zscore=False, apply_guassian_filter=False, source_path=source_path)
last_t_vr = clusters_vr[clusters_vr.index[0]].count(
    bin_size=time_bs, time_units='ms').index[-1]
ep_vr = nap.IntervalSet(start=0, end=last_t_vr, time_units='s')

# Classify cells — needed for optimal_shift before loading OF1
gcs, ngs, all_cells = classify_cells_both_sessions(mouse, day, source_path=source_path)
gc_ids = set(gcs.cluster_id.values.astype(int))

# Optimal shift: aligns spikes to the lagged position in OF1
optimal_shift = float(gcs['travel'].iloc[0]) if 'travel' in gcs.columns else 0.0
print(f'Optimal shift: {optimal_shift:.2f} cm')

print('Loading OF1...')
tcs_of, tcs_time_of, beh_of, clusters_of, ep_of = compute_of_tcs(
    mouse, day, apply_zscore=False, apply_guassian_filter=False,
    fill_nans_from_neighbors=True,
    source_path=source_path, session='OF1',
    optimal_shift=optimal_shift)

# All GC cells EXCEPT the target
cov_gc_ids = sorted([c for c in gc_ids if c != TARGET_ID and c in tcs_time_vr])
print(f'VR: {len(tcs_time_vr)} cells  |  OF1: {len(tcs_time_of)} cells')
print(f'GC covariate cells (excl. target): {len(cov_gc_ids)}')
assert TARGET_ID in tcs_time_vr,  f'Target {TARGET_ID} not found in VR tcs_time'
assert TARGET_ID in tcs_time_of,  f'Target {TARGET_ID} not found in OF1 tcs_time'

## 2. Extract behavioural signals

In [ ]:
# ── VR ────────────────────────────────────────────────────────────────────────
y_vr  = np.array(tcs_time_vr[TARGET_ID])
T_vr  = len(y_vr)

def _bin_vr(key, T):
    a = np.array(beh_vr[key].bin_average(bin_size=time_bs, time_units='ms', ep=ep_vr))
    return pd.Series(a).ffill().bfill().values[:T]

dt_vr  = _bin_vr('travel', T_vr) - ((beh_vr['trial_number'][0] - 1) * tl)
pos_vr = dt_vr % tl

# Build GC covariate matrix for VR
def _pad(arr, T):
    arr = np.array(arr)[:T]
    return np.pad(arr, (0, max(0, T - len(arr))))

gc_mat_vr = np.vstack([_pad(np.array(tcs_time_vr[c]), T_vr)
                       for c in cov_gc_ids]).T   # (T, n_gc)
x_vr = np.column_stack([pos_vr, gc_mat_vr])     # pos + all GC cells
print(f'VR covariate matrix: {x_vr.shape}  (T × [1 pos + {len(cov_gc_ids)} GC])')

# ── OF1 ───────────────────────────────────────────────────────────────────────
y_of  = np.array(tcs_time_of[TARGET_ID])
T_of  = len(y_of)
cov_gc_ids_of = [c for c in cov_gc_ids if c in tcs_time_of]

def _bin_of(key, T):
    a = np.array(beh_of[key].bin_average(bin_size=time_bs, time_units='ms', ep=ep_of))
    return pd.Series(a).ffill().bfill().values[:T]

px_of = _bin_of('head_x', T_of)
py_of = _bin_of('head_y', T_of)

gc_mat_of = np.vstack([_pad(np.array(tcs_time_of[c]), T_of)
                       for c in cov_gc_ids_of]).T
x_of = np.column_stack([px_of, py_of, gc_mat_of])
print(f'OF1 covariate matrix: {x_of.shape}  (T × [2 pos + {len(cov_gc_ids_of)} GC])')

## 3. Fit XGBoost for each history length
For VR: `pos + all_GC_cells`, for OF1: `pos_x, pos_y + all_GC_cells`.
Each history length sets both `max_time` and `n_filters = max_time / time_bs`.

In [ ]:
results_vr = {}   # key → {'Y_hat': ..., 'pR2': ...}
results_of = {}

# ── No-history: cov_history=False — only current time bin, no lag features ────
print('Fitting no-history model (current time bin only)...')
xgb_nohist = MLencoding(tunemodel='xgboost', cov_history=False, spike_history=False,
                         window=time_bs, n_filters=1, max_time=time_bs)

Y_hat_nh_vr, pr2_nh_vr = xgb_nohist.fit_cv(x_vr, y_vr, verbose=0,
                                              continuous_folds=True, n_cv=N_CV)
results_vr['no_history'] = {'Y_hat': Y_hat_nh_vr, 'pR2': float(np.nanmean(pr2_nh_vr)),
                             'label': 'No history\n(current bin)'}
print(f'  VR  pR² = {results_vr["no_history"]["pR2"]:+.4f}')

Y_hat_nh_of, pr2_nh_of = xgb_nohist.fit_cv(x_of, y_of, verbose=0,
                                              continuous_folds=True, n_cv=N_CV)
results_of['no_history'] = {'Y_hat': Y_hat_nh_of, 'pR2': float(np.nanmean(pr2_nh_of)),
                             'label': 'No history\n(current bin)'}
print(f'  OF1 pR² = {results_of["no_history"]["pR2"]:+.4f}')

# ── History length sweep ───────────────────────────────────────────────────────
for hl in HISTORY_LENGTHS:
    xgb = MLencoding(tunemodel='xgboost', cov_history=True, spike_history=False,
                     window=time_bs, n_filters=FIXED_NFILTERS, max_time=hl)
    print(f'\nHistory = {hl} ms  (nfilters=FIXED={FIXED_NFILTERS})')

    Y_hat_vr, pr2_vr = xgb.fit_cv(x_vr, y_vr, verbose=0,
                                    continuous_folds=True, n_cv=N_CV)
    results_vr[hl] = {'Y_hat': Y_hat_vr, 'pR2': float(np.nanmean(pr2_vr)),
                      'label': f'{hl} ms'}
    print(f'  VR  pR² = {results_vr[hl]["pR2"]:+.4f}')

    Y_hat_of, pr2_of = xgb.fit_cv(x_of, y_of, verbose=0,
                                    continuous_folds=True, n_cv=N_CV)
    results_of[hl] = {'Y_hat': Y_hat_of, 'pR2': float(np.nanmean(pr2_of)),
                      'label': f'{hl} ms'}
    print(f'  OF1 pR² = {results_of[hl]["pR2"]:+.4f}')

print('\nDone fitting.')
print('\nSummary:')
print(f'  {"Condition":22s}  VR pR²    OF1 pR²')
for k in ['no_history'] + HISTORY_LENGTHS:
    lbl = results_vr[k]['label'].replace('\\n', ' ')
    print(f'  {lbl:22s}  {results_vr[k]["pR2"]:+.4f}    {results_of[k]["pR2"]:+.4f}')

## 4. Reconstruct rate maps
Uses `compute_vr_tcs_using_expected_spikes` and `compute_of_tcs_using_expected_spikes`
to bin the XGBoost predicted spike probabilities (Y_hat) back into proper
trial × position (VR) and 2D position (OF1) rate maps.

In [ ]:
from spatial_manifolds.detect_grids import _fill_nans_from_neighbors
from spatial_manifolds.tuning_scores.grid_score import autocorr2d

# Pseudo cluster IDs for reconstruction (cells not involved in the analysis)
_excl    = {TARGET_ID} | set(cov_gc_ids)
ALL_KEYS = ['no_history'] + HISTORY_LENGTHS  # ordered for pseudo ID mapping
_pseudo  = [c for c in all_cells.cluster_id.values.astype(int) if c not in _excl]
assert len(_pseudo) >= len(ALL_KEYS), 'Not enough pseudo IDs'

# ── VR rate map reconstruction ────────────────────────────────────────────────
exp_spk_vr = {_pseudo[j]: results_vr[k]['Y_hat']
               for j, k in enumerate(ALL_KEYS)}

print('Reconstructing VR rate maps...')
tcs_recon_vr, _, _, last_bin_vr, _, _ = compute_vr_tcs_using_expected_spikes(
    mouse, day, apply_zscore=False, vr_type='VR',
    source_path=source_path, expected_spikes=exp_spk_vr)

rm_true_vr = gaussian_filter(
    np.nan_to_num(tcs_vr[TARGET_ID]).astype(np.float64), sigma=SIGMA)[:last_bin_vr]
rm_pred_vr = {
    k: gaussian_filter(
        np.nan_to_num(tcs_recon_vr[_pseudo[j]]).astype(np.float64), sigma=SIGMA
    )[:last_bin_vr]
    for j, k in enumerate(ALL_KEYS)
}

# ── OF1 rate map reconstruction ───────────────────────────────────────────────
# Pass optimal_shift so position is lagged to match VR alignment
exp_spk_of = {_pseudo[j]: results_of[k]['Y_hat']
               for j, k in enumerate(ALL_KEYS)}

print('Reconstructing OF1 rate maps...')
tcs_recon_of, _, _, _, _ = compute_of_tcs_using_expected_spikes(
    mouse, day, apply_zscore=False, apply_guassian_filter=True,
    source_path=source_path, expected_spikes=exp_spk_of,
    optimal_shift=optimal_shift)

# True OF1 map: already smoothed and fill_nans applied via compute_of_tcs above
rm_true_of = gaussian_filter(
    np.nan_to_num(tcs_of[TARGET_ID]).astype(np.float64), sigma=SIGMA)

# Reconstructed maps: apply fill_nans_from_neighbors to handle unvisited bins,
# matching the treatment of the true rate map in compute_of_tcs
rm_pred_of = {}
for j, k in enumerate(ALL_KEYS):
    raw = np.nan_to_num(tcs_recon_of[_pseudo[j]]).astype(np.float64)
    raw[raw == 0] = np.nan          # treat unvisited zeros as NaN before filling
    filled = _fill_nans_from_neighbors(raw)
    rm_pred_of[k] = gaussian_filter(np.nan_to_num(filled), sigma=SIGMA)

# ── Spatial autocorrelograms ──────────────────────────────────────────────────
print('Computing autocorrelograms (this may take a moment)...')
ac_true_of = autocorr2d(rm_true_of)
ac_pred_of  = {k: autocorr2d(rm_pred_of[k]) for k in ALL_KEYS}
print('Done.')

## 5. pR² vs history length

In [ ]:
fig_pr2, ax = plt.subplots(figsize=(6, 3.5))

pr2_vr_vals = [results_vr[hl]['pR2'] for hl in HISTORY_LENGTHS]
pr2_of_vals = [results_of[hl]['pR2'] for hl in HISTORY_LENGTHS]

# History sweep lines
ax.plot(HISTORY_LENGTHS, pr2_vr_vals, 'o-', color=COL_GC,    lw=2, label='VR',  zorder=3)
ax.plot(HISTORY_LENGTHS, pr2_of_vals, 's-', color='#3171ae',  lw=2, label='OF1', zorder=3)

# No-history points — shown as separate markers at a fixed x position to the left
# (they use cov_history=False so 'history = 0' is conceptually correct)
x_nohist = HISTORY_LENGTHS[0] * 0.4   # position left of the first history point
ax.scatter([x_nohist], [results_vr['no_history']['pR2']],
           marker='D', s=70, color=COL_GC,   zorder=5, label='VR (no history)')
ax.scatter([x_nohist], [results_of['no_history']['pR2']],
           marker='D', s=70, color='#3171ae', zorder=5, label='OF1 (no history)')
ax.axvline(x_nohist, color='#cccccc', lw=0.8, ls=':', zorder=1)
ax.text(x_nohist, ax.get_ylim()[0] if ax.get_ylim()[0] != 0 else -0.005,
        'No\nhist.', ha='center', va='top', fontsize=7, color='#666666')

ax.set_xscale('log')
ax.set_xlabel('History length (ms)  [log scale]', fontsize=10)
ax.set_ylabel('pR²', fontsize=10)
ax.set_title(
    f'M{mouse} D{day}  GC {TARGET_ID}  —  pos + all GC covariates',
    fontsize=9
)
ax.axhline(0, color='#cccccc', lw=0.8, ls='--')
ax.set_xticks(HISTORY_LENGTHS)
ax.set_xticklabels([str(h) for h in HISTORY_LENGTHS])
ax.legend(fontsize=8, frameon=False, ncol=2)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig(fig_path + f'history_pr2_M{mouse}D{day}_GC{TARGET_ID}.pdf',
            bbox_inches='tight', dpi=150)
plt.show()

## 6. Rate map figure — VR and OF1 across history lengths
Each column is one history length. Column 0 is the true rate map.
Row 0 = VR (trial × position), Row 1 = OF1 (2D open field).
pR² is annotated above each predicted panel.

In [ ]:
ALL_KEYS = ['no_history'] + HISTORY_LENGTHS
n_cols   = len(ALL_KEYS) + 1
rng_fig  = np.random.default_rng(42)

# ── Valid timepoint mask: speed ≤ 50 cm/s and non-outlier position ───────────
SPEED_THRESH = 50.0   # cm/s
spd_mask  = spd_of[:T_of] <= SPEED_THRESH
# Position outlier mask: within 1st–99th percentile of x and y
x_lo, x_hi = np.nanpercentile(px_of[:T_of], 1), np.nanpercentile(px_of[:T_of], 99)
y_lo, y_hi = np.nanpercentile(py_of[:T_of], 1), np.nanpercentile(py_of[:T_of], 99)
pos_mask  = ((px_of[:T_of] >= x_lo) & (px_of[:T_of] <= x_hi) &
             (py_of[:T_of] >= y_lo) & (py_of[:T_of] <= y_hi))
valid_mask = spd_mask & pos_mask
print(f'Valid bins: {valid_mask.sum()}/{T_of}  '
      f'({100*valid_mask.mean():.1f}%)  '
      f'[speed>{SPEED_THRESH} or position outlier removed]')

# ── Poisson-sample predicted spikes and build OF rate maps ───────────────────
# For each model: draw spike counts from Poisson(Y_hat), then build the 2D
# rate map from those sampled spike positions. This makes the trajectory plot
# and the rate map show the SAME predicted spikes.
def poisson_rate_map(Y_hat, px, py, T, n_bins=40, sigma=2.5, rng=None, mask=None):
    """Sample spikes from Poisson(Y_hat), return (spike_x, spike_y, rate_map).
    mask: boolean array (T,) — only use valid timepoints (speed/position filtered).
    """
    if rng is None: rng = np.random.default_rng()
    T_ = min(T, len(Y_hat), len(px), len(py))
    if mask is not None:
        m = mask[:T_]
    else:
        m = np.ones(T_, dtype=bool)
    cnt     = rng.poisson(np.maximum(Y_hat[:T_], 0))
    cnt_m   = cnt * m          # zero out invalid bins
    sx      = np.repeat(px[:T_][m], cnt[m])
    sy      = np.repeat(py[:T_][m], cnt[m])
    xe = np.linspace(np.nanpercentile(px[:T_][m], 1),
                     np.nanpercentile(px[:T_][m], 99), n_bins+1)
    ye = np.linspace(np.nanpercentile(py[:T_][m], 1),
                     np.nanpercentile(py[:T_][m], 99), n_bins+1)
    tc,  _, _ = np.histogram2d(sx, sy, bins=[xe, ye])
    occ, _, _ = np.histogram2d(px[:T_][m], py[:T_][m], bins=[xe, ye])
    rm = tc / np.where(occ > 0, occ, np.nan)
    rm = gaussian_filter(np.nan_to_num(_fill_nans_from_neighbors(rm.T)), sigma=sigma)
    return sx, sy, rm

pred_spk_x, pred_spk_y, rm_pred_poisson = {}, {}, {}
for k in ALL_KEYS:
    sx, sy, rm = poisson_rate_map(results_of[k]['Y_hat'], px_of, py_of, T_of,
                                   rng=rng_fig, mask=valid_mask)
    pred_spk_x[k], pred_spk_y[k], rm_pred_poisson[k] = sx, sy, rm

# True spike positions
true_spike_mask = (y_of[:T_of] > 0) & valid_mask
true_spk_x = px_of[:T_of][true_spike_mask]
true_spk_y = py_of[:T_of][true_spike_mask]

# Autocorrelograms from Poisson-sampled rate maps
print('Computing autocorrelograms...')
ac_pred_poisson = {k: autocorr2d(rm_pred_poisson[k]) for k in ALL_KEYS}
print('Done.')

# ── Figure: 4 rows ────────────────────────────────────────────────────────────
fig, axes = plt.subplots(
    4, n_cols,
    figsize=(n_cols * 2.0, 12.5),
    gridspec_kw={'hspace': 0.30, 'wspace': 0.06,
                 'height_ratios': [2.8, 1.5, 1.5, 1.5]},
)

CMAP_VR    = white_to_hex_cmap(COL_GC)
CMAP_OF    = 'viridis'
CMAP_AC    = 'viridis'
COL_HEADER = {k: ('#888888' if k == 'no_history' else COL_GC) for k in ALL_KEYS}

# ── Row 0: VR rate maps ───────────────────────────────────────────────────────
ax = axes[0, 0]
plot_firing_rate_map(ax, rm_true_vr, bs=bs, tl=tl, p=95, cmap=CMAP_VR)
ax.set_title('True', fontsize=9, fontweight='bold')
ax.set_ylabel('VR\nTrial', fontsize=9, labelpad=3)
ax.tick_params(labelsize=7)
for ci, k in enumerate(ALL_KEYS):
    ax = axes[0, ci + 1]
    plot_firing_rate_map(ax, rm_pred_vr[k], bs=bs, tl=tl, p=95, cmap=CMAP_VR)
    ax.set_title(f'pR²={results_vr[k]["pR2"]:+.3f}', fontsize=8, color=COL_HEADER[k])
    ax.set_yticks([]); ax.tick_params(labelsize=7)

# ── Row 1: OF1 trajectory + spikes ───────────────────────────────────────────
# True: grey trace + black scatter at spike locations
ax = axes[1, 0]
# Mask trajectory: NaN at invalid bins so line breaks rather than spans arena
px_masked = np.where(valid_mask, px_of[:T_of], np.nan)
py_masked = np.where(valid_mask, py_of[:T_of], np.nan)
ax.plot(px_masked, py_masked, color='#cccccc', lw=0.3, alpha=0.6, zorder=1)
ax.scatter(true_spk_x, true_spk_y, color='black', s=2, alpha=0.6, linewidths=0, zorder=2)
ax.set_aspect('equal'); ax.set_xticks([]); ax.set_yticks([])
ax.spines[:].set_visible(False)
ax.set_title('True spikes', fontsize=8, fontweight='bold')
ax.set_ylabel('OF1\nTrajectory', fontsize=9, labelpad=3)

# Predicted: grey trace + coloured scatter at Poisson-sampled spike locations
for ci, k in enumerate(ALL_KEYS):
    ax = axes[1, ci + 1]
    ax.plot(px_masked, py_masked, color='#cccccc', lw=0.3, alpha=0.6, zorder=1)
    ax.scatter(pred_spk_x[k], pred_spk_y[k],
               color='black', s=2, alpha=0.6, linewidths=0, zorder=2)
    ax.set_aspect('equal'); ax.set_xticks([]); ax.set_yticks([])
    ax.spines[:].set_visible(False)
    ax.set_title(f'Pred. spikes ({len(pred_spk_x[k])})', fontsize=7)

# ── Row 2: OF1 rate maps — per-map 0–99th percentile ─────────────────────────
ax = axes[2, 0]
vmin_true = np.nanmin(rm_true_of)
vmax_true = np.nanpercentile(rm_true_of, 99)
ax.imshow(rm_true_of, origin='lower', cmap=CMAP_OF,
          vmin=vmin_true, vmax=vmax_true, interpolation='nearest')
ax.set_ylabel('OF1\nRate map', fontsize=9, labelpad=3)
ax.set_title('True', fontsize=9, fontweight='bold')
ax.set_xticks([]); ax.set_yticks([])

for ci, k in enumerate(ALL_KEYS):
    ax    = axes[2, ci + 1]
    rm    = rm_pred_poisson[k]
    vmax_k = np.nanpercentile(rm, 99)
    ax.imshow(rm, origin='lower', cmap=CMAP_OF,
              vmin=0, vmax=vmax_k, interpolation='nearest')
    ax.set_title(f'pR²={results_of[k]["pR2"]:+.3f}', fontsize=8, color=COL_HEADER[k])
    ax.set_xticks([]); ax.set_yticks([])

# ── Row 3: Autocorrelograms — viridis, per-map 0–99th percentile ─────────────
ax = axes[3, 0]
vmax_ac_true = np.nanpercentile(ac_true_of[~np.isnan(ac_true_of)], 99)
ax.imshow(ac_true_of, origin='lower', cmap=CMAP_AC,
          vmin=0, vmax=vmax_ac_true, interpolation='nearest')
ax.set_ylabel('OF1\nAutocorr', fontsize=9, labelpad=3)
ax.set_title('True', fontsize=9, fontweight='bold')
ax.set_xticks([]); ax.set_yticks([])

for ci, k in enumerate(ALL_KEYS):
    ax     = axes[3, ci + 1]
    ac     = ac_pred_poisson[k]
    vmax_ac = np.nanpercentile(ac[~np.isnan(ac)], 99)
    ax.imshow(ac, origin='lower', cmap=CMAP_AC,
              vmin=0, vmax=vmax_ac, interpolation='nearest')
    ax.set_xticks([]); ax.set_yticks([])

# ── Column headers ────────────────────────────────────────────────────────────
for ci, k in enumerate(ALL_KEYS):
    label = 'No history\n(current bin)' if k == 'no_history' else f'History\n{k} ms'
    axes[0, ci+1].text(0.5, 1.16, label,
                       transform=axes[0, ci+1].transAxes,
                       ha='center', va='bottom', fontsize=8.0, fontweight='bold',
                       color=COL_HEADER[k])

fig.suptitle(
    f'M{mouse} D{day}  GC {TARGET_ID}  |  '
    f'pos + {len(cov_gc_ids)} GC covariate cells  |  '
    f'Optimal shift = {optimal_shift:.1f} cm',
    fontsize=10, fontweight='bold', y=1.01
)
save_path = fig_path + f'history_rate_maps_M{mouse}D{day}_GC{TARGET_ID}.pdf'
fig.savefig(save_path, bbox_inches='tight', dpi=300)
plt.show()
print(f'Saved → {save_path}')


## 7. Circular shuffle validation

Tests whether positive Δ pR² genuinely reflects coordination by comparing
it against a null distribution from circular shuffles of the covariate spike trains.

**Circular shuffle**: the GC covariate matrix is rolled along the time axis by a
random offset drawn uniformly from ±[min_shift, session_length/2]. This preserves
the autocorrelation and firing-rate envelope of each covariate cell but destroys
moment-to-moment co-activity with the target.

The baseline (position-only pR²) is fitted once. For each shuffle, only the
full model (pos + shuffled GC cells) needs refitting, making it faster.

**Δ pR² = pR²(pos + GC cells) − pR²(pos only)**

If the real Δ pR² exceeds the 95th percentile of the shuffle null distribution,
the coordination signal is significant.

In [ ]:
import time as _time

# ── Config ───────────────────────────────────────────────────────────────────
N_SHUFFLES   = 200        # number of circular shifts
MIN_SHIFT_S  = 30.0       # minimum shift in seconds (avoids near-zero offsets)
SHUF_HL      = 1000       # history length to use for shuffle validation (ms)
                          # should match a history tested in HISTORY_LENGTHS

assert SHUF_HL in HISTORY_LENGTHS, f'{SHUF_HL} ms not in HISTORY_LENGTHS — rerun fitting cells first'

# ── Build position-only baseline covariate arrays ─────────────────────────────
x_pos_vr = pos_vr[:, None]                          # (T_vr, 1)
x_pos_of = np.column_stack([px_of, py_of])          # (T_of, 2)

xgb_shuf = MLencoding(tunemodel='xgboost', cov_history=True, spike_history=False,
                       window=time_bs, n_filters=FIXED_NFILTERS, max_time=SHUF_HL)

print(f'History length for shuffle: {SHUF_HL} ms  (nfilters=FIXED={FIXED_NFILTERS})')
print(f'N shuffles: {N_SHUFFLES}')

# ── Fit baseline (pos only) once ─────────────────────────────────────────────
print('Fitting pos-only baselines...')
_, pr2_bl_vr = xgb_shuf.fit_cv(x_pos_vr, y_vr, verbose=0, continuous_folds=True, n_cv=N_CV)
_, pr2_bl_of = xgb_shuf.fit_cv(x_pos_of, y_of, verbose=0, continuous_folds=True, n_cv=N_CV)
pr2_bl_vr_mean = float(np.nanmean(pr2_bl_vr))
pr2_bl_of_mean = float(np.nanmean(pr2_bl_of))
print(f'  VR  baseline pR² = {pr2_bl_vr_mean:+.4f}')
print(f'  OF1 baseline pR² = {pr2_bl_of_mean:+.4f}')

# Real Δ pR² (already fitted above)
real_delta_vr = results_vr[SHUF_HL]['pR2'] - pr2_bl_vr_mean
real_delta_of = results_of[SHUF_HL]['pR2'] - pr2_bl_of_mean
print(f'\nReal Δ pR² VR  = {real_delta_vr:+.4f}')
print(f'Real Δ pR² OF1 = {real_delta_of:+.4f}')

In [ ]:
# ── Circular shuffle loop ─────────────────────────────────────────────────────
rng_shuf = np.random.default_rng(42)

min_shift_bins_vr = int(MIN_SHIFT_S * 1000 / time_bs)
min_shift_bins_of = int(MIN_SHIFT_S * 1000 / time_bs)

shuf_delta_vr = np.full(N_SHUFFLES, np.nan)
shuf_delta_of = np.full(N_SHUFFLES, np.nan)

t0 = _time.time()
for si in range(N_SHUFFLES):
    # ── Sample a circular shift ────────────────────────────────────────────────
    # Draw from [min_shift, T/2] then randomly negate for both directions
    shift_vr = rng_shuf.integers(min_shift_bins_vr, T_vr // 2)
    if rng_shuf.random() < 0.5: shift_vr = -shift_vr

    shift_of = rng_shuf.integers(min_shift_bins_of, T_of // 2)
    if rng_shuf.random() < 0.5: shift_of = -shift_of

    # ── Roll GC covariate matrix ──────────────────────────────────────────────
    # All covariate cells shifted by the same offset (standard global shift)
    gc_mat_vr_shuf = np.roll(gc_mat_vr, shift_vr, axis=0)
    gc_mat_of_shuf = np.roll(gc_mat_of, shift_of, axis=0)

    x_shuf_vr = np.column_stack([pos_vr, gc_mat_vr_shuf])
    x_shuf_of = np.column_stack([px_of, py_of, gc_mat_of_shuf])

    # ── Fit full model with shuffled covariates ───────────────────────────────
    _, pr2_s_vr = xgb_shuf.fit_cv(x_shuf_vr, y_vr, verbose=0, continuous_folds=True, n_cv=N_CV)
    _, pr2_s_of = xgb_shuf.fit_cv(x_shuf_of, y_of, verbose=0, continuous_folds=True, n_cv=N_CV)

    shuf_delta_vr[si] = float(np.nanmean(pr2_s_vr)) - pr2_bl_vr_mean
    shuf_delta_of[si] = float(np.nanmean(pr2_s_of)) - pr2_bl_of_mean

    elapsed = _time.time() - t0
    eta     = elapsed / (si + 1) * (N_SHUFFLES - si - 1)
    print(f'  Shuffle {si+1:3d}/{N_SHUFFLES}  '
          f'VR Δ={shuf_delta_vr[si]:+.4f}  '
          f'OF Δ={shuf_delta_of[si]:+.4f}  '
          f'ETA={eta:.0f}s', end='\r')

print(f'\nDone in {_time.time()-t0:.1f}s')

# ── Percentile p-values ───────────────────────────────────────────────────────
p_vr = np.mean(shuf_delta_vr >= real_delta_vr)
p_of = np.mean(shuf_delta_of >= real_delta_of)
pct95_vr = np.nanpercentile(shuf_delta_vr, 95)
pct95_of = np.nanpercentile(shuf_delta_of, 95)
print(f'\nVR:  real Δ pR²={real_delta_vr:+.4f}  95th pctile={pct95_vr:+.4f}  p={p_vr:.3f}')
print(f'OF1: real Δ pR²={real_delta_of:+.4f}  95th pctile={pct95_of:+.4f}  p={p_of:.3f}')

In [ ]:
# ── Figure: shuffle null distribution ────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(10, 4),
                          gridspec_kw={'wspace': 0.35})

for ax, shuf_delta, real_delta, pct95, p_val, session, color in [
    (axes[0], shuf_delta_vr, real_delta_vr, pct95_vr, p_vr, 'VR',  COL_GC),
    (axes[1], shuf_delta_of, real_delta_of, pct95_of, p_of, 'OF1', '#3171ae'),
]:
    # Histogram of shuffle null
    ax.hist(shuf_delta, bins=30, color='#aaaaaa', alpha=0.75,
            edgecolor='white', linewidth=0.5, label='Circular shuffle')

    # 95th percentile line
    ax.axvline(pct95, color='#555555', lw=1.5, ls='--',
               label=f'95th pctile ({pct95:+.4f})')

    # Real Δ pR² line
    ax.axvline(real_delta, color=color, lw=2.5, zorder=5,
               label=f'Real Δ pR² ({real_delta:+.4f})')

    # Zero reference
    ax.axvline(0, color='k', lw=0.8, ls=':', alpha=0.5)

    # p-value annotation
    sig = '***' if p_val < 0.001 else ('**' if p_val < 0.01 else
           ('*' if p_val < 0.05 else 'ns'))
    ax.text(0.97, 0.97,
            f'p = {p_val:.3f}  {sig}\n'
            f'n = {N_SHUFFLES} shuffles\n'
            f'shift ≥ {MIN_SHIFT_S:.0f} s',
            transform=ax.transAxes, fontsize=8,
            ha='right', va='top', color='#333333',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
                      edgecolor='#cccccc', linewidth=0.8))

    ax.set_xlabel('Δ pR²  (pos + GC cells) − (pos only)', fontsize=9)
    ax.set_ylabel('Count', fontsize=9)
    ax.set_title(
        f'{session} — {SHUF_HL} ms history\n'
        f'Target GC {TARGET_ID},  {len(cov_gc_ids)} GC covariates',
        fontsize=9, fontweight='bold'
    )
    ax.legend(fontsize=8, frameon=False)
    ax.spines[['top', 'right']].set_visible(False)
    ax.tick_params(labelsize=8)

fig.suptitle(
    f'M{mouse} D{day}  — Circular shuffle validation of Δ pR² coordination measure',
    fontsize=10, fontweight='bold'
)
save_shuf = (fig_path +
             f'shuffle_validation_M{mouse}D{day}_GC{TARGET_ID}_h{SHUF_HL}.pdf')
fig.savefig(save_shuf, bbox_inches='tight', dpi=300)
plt.show()
print(f'Saved → {save_shuf}')

## n_filters × history sweep
For each history length, sweep n_filters from 1 to 10 and plot pR².
Shows how temporal resolution (n_filters) interacts with window size (history length).

In [ ]:
import time as _time

NF_RANGE       = list(range(1, 11))   # 1 to 10
HL_COLORS_VR   = plt.cm.Reds(  np.linspace(0.35, 0.9, len(HISTORY_LENGTHS)))
HL_COLORS_OF   = plt.cm.Blues( np.linspace(0.35, 0.9, len(HISTORY_LENGTHS)))

# ── Fit all (history, nfilters) combinations ─────────────────────────────────
pr2_grid_vr = {}   # (hl, nf) → pR²
pr2_grid_of = {}

n_total = len(HISTORY_LENGTHS) * len(NF_RANGE)
done    = 0
t0      = _time.time()

for hl in HISTORY_LENGTHS:
    for nf in NF_RANGE:
        xgb = MLencoding(tunemodel='xgboost', cov_history=True, spike_history=False,
                          window=time_bs, n_filters=nf, max_time=hl)

        _, pr2_vr = xgb.fit_cv(x_vr, y_vr, verbose=0,
                                continuous_folds=True, n_cv=N_CV)
        _, pr2_of = xgb.fit_cv(x_of, y_of, verbose=0,
                                continuous_folds=True, n_cv=N_CV)

        pr2_grid_vr[(hl, nf)] = float(np.nanmean(pr2_vr))
        pr2_grid_of[(hl, nf)] = float(np.nanmean(pr2_of))

        done += 1
        el   = _time.time() - t0
        eta  = el / done * (n_total - done)
        print(f'  hl={hl:4d}ms  nf={nf:2d}  '
              f'VR={pr2_grid_vr[(hl,nf)]:+.4f}  '
              f'OF={pr2_grid_of[(hl,nf)]:+.4f}  '
              f'ETA={eta:.0f}s', end='\r')

print(f'\nDone in {_time.time()-t0:.1f}s')


In [ ]:
# ── Plot: pR² vs n_filters, one line per history length ───────────────────────
fig, axes = plt.subplots(1, 2, figsize=(11, 4),
                          gridspec_kw={'wspace': 0.30})

for ax, pr2_grid, colors, session, ylbl in [
    (axes[0], pr2_grid_vr, HL_COLORS_VR, 'VR',  'pR²'),
    (axes[1], pr2_grid_of, HL_COLORS_OF, 'OF1', 'pR²'),
]:
    for hl, col in zip(HISTORY_LENGTHS, colors):
        y = [pr2_grid[(hl, nf)] for nf in NF_RANGE]
        ax.plot(NF_RANGE, y, 'o-', color=col, lw=2, ms=6,
                label=f'{hl} ms')
        # Mark the peak nfilters
        best_nf = NF_RANGE[int(np.argmax(y))]
        ax.scatter([best_nf], [max(y)], marker='*', s=120,
                   color=col, zorder=5)

    # Mark FIXED_NFILTERS=5
    ax.axvline(FIXED_NFILTERS, color='#888888', lw=1.2, ls='--', alpha=0.7,
               label=f'Fixed n_filters={FIXED_NFILTERS}')

    ax.axhline(0, color='#cccccc', lw=0.8, ls=':')
    ax.set_xlabel('n_filters', fontsize=10)
    ax.set_ylabel(ylbl, fontsize=10)
    ax.set_title(f'{session} — GC {TARGET_ID},  {len(cov_gc_ids)} GC covariates',
                 fontsize=9)
    ax.set_xticks(NF_RANGE)
    ax.legend(title='History', fontsize=8, frameon=False,
              title_fontsize=8, loc='lower right')
    ax.spines[['top', 'right']].set_visible(False)
    ax.tick_params(labelsize=8)

fig.suptitle(
    f'pR² vs n_filters for each history length — M{mouse} D{day}\n'
    f'★ = peak n_filters per history  |  dashed = fixed n_filters={FIXED_NFILTERS}',
    fontsize=9, fontweight='bold'
)
plt.tight_layout()
save_nf_hl = (fig_path +
              f'nfilters_history_grid_M{mouse}D{day}_GC{TARGET_ID}.pdf')
fig.savefig(save_nf_hl, bbox_inches='tight', dpi=200)
plt.show()
print(f'Saved → {save_nf_hl}')


---
## n_filters sweep — fixed history 1000 ms

With history length fixed at 1000 ms, sweeps `n_filters` from 1 to 20 to show
how temporal resolution of the filter bank affects prediction quality.

**Panel layout:**
1. Filter bank visualisation — how the raised cosine filters spread across the
   1000 ms window at n_filters = 1, 5, 10, 20
2. pR² vs n_filters (VR and OF1)
3. VR rate maps at selected n_filters
4. OF1 rate maps + autocorrelograms at selected n_filters

In [ ]:
# ── Visualise raised cosine filter banks at different n_filters ───────────────
from spatial_manifolds.mlencoding import MLencoding

NFILTERS_SHOW  = [1, 5, 10, 20]   # n_filters values to illustrate
NFILTERS_SWEEP = list(range(1, 21))  # 1 → 20
HISTORY_NF     = 1000              # ms — fixed for this assay

def get_filters(max_time, nfilt, window=time_bs):
    """Return (t_ms, filter_array) where filter_array is (nfilt, n_bins)."""
    n_bins = int(max_time / window)
    t      = np.linspace(0, max_time, n_bins)
    nt     = np.log(t + 0.1)
    cSt, cEnd = nt[1], nt[-1]
    db    = (cEnd - cSt) / nfilt
    c     = np.arange(cSt, cEnd, db)
    filts = []
    for k in range(nfilt):
        f = (np.cos(np.maximum(-np.pi, np.minimum(np.pi,
                    (nt - c[k]) * np.pi / db))) + 1) / 2
        f = f / (np.sum(f) + 1e-12)
        filts.append(f)
    return t, np.array(filts)

# ── Figure: filter banks ──────────────────────────────────────────────────────
fig_filt, axes_filt = plt.subplots(
    1, len(NFILTERS_SHOW), figsize=(13, 2.8),
    gridspec_kw={'wspace': 0.12}
)
peak_times_all = {}

for ax, nf in zip(axes_filt, NFILTERS_SHOW):
    t_ms, filts = get_filters(HISTORY_NF, nf)
    cols = plt.cm.viridis(np.linspace(0.15, 0.9, nf))
    peaks = []
    for k, (f, c) in enumerate(zip(filts, cols)):
        ax.plot(t_ms, f, color=c, lw=1.8)
        peak_t = t_ms[np.argmax(f)]
        peaks.append(peak_t)
        ax.axvline(peak_t, color=c, lw=0.6, ls='--', alpha=0.5)
    peak_times_all[nf] = peaks
    ax.set_title(f'n_filters = {nf}\npeaks: {[f"{p:.0f}" for p in peaks]} ms',
                 fontsize=8)
    ax.set_xlabel('Time lag (ms)', fontsize=8)
    ax.set_xlim(0, HISTORY_NF)
    ax.set_ylim(bottom=0)
    ax.spines[['top', 'right']].set_visible(False)
    ax.tick_params(labelsize=7)
    if ax is axes_filt[0]:
        ax.set_ylabel('Filter weight', fontsize=8)
    else:
        ax.set_yticklabels([])

fig_filt.suptitle(
    f'Raised cosine filter banks — history = {HISTORY_NF} ms, time_bs = {time_bs} ms',
    fontsize=9, fontweight='bold'
)
fig_filt.savefig(fig_path + f'filter_banks_M{mouse}D{day}.pdf', bbox_inches='tight', dpi=200)
plt.show()
print('Filter peak times (ms):')
for nf, peaks in peak_times_all.items():
    print(f'  nfilters={nf:2d}: {[f"{p:.0f}" for p in peaks]}')

In [ ]:
# ── Sweep n_filters 1-20 at fixed history 1000 ms ────────────────────────────
results_nf_vr = {}   # nfilters → {'Y_hat': ..., 'pR2': ...}
results_nf_of = {}

# Covariate matrices at T_vr and T_of (already built above)
print(f'Sweeping n_filters {NFILTERS_SWEEP} at history={HISTORY_NF} ms')
print(f'VR covariate matrix: {x_vr.shape}  |  OF1: {x_of.shape}')

for nf in NFILTERS_SWEEP:
    xgb_nf = MLencoding(tunemodel='xgboost', cov_history=True, spike_history=False,
                         window=time_bs, n_filters=nf, max_time=HISTORY_NF)

    Y_hat_vr, pr2_vr = xgb_nf.fit_cv(x_vr, y_vr, verbose=0,
                                       continuous_folds=True, n_cv=N_CV)
    results_nf_vr[nf] = {'Y_hat': Y_hat_vr, 'pR2': float(np.nanmean(pr2_vr))}

    Y_hat_of, pr2_of = xgb_nf.fit_cv(x_of, y_of, verbose=0,
                                       continuous_folds=True, n_cv=N_CV)
    results_nf_of[nf] = {'Y_hat': Y_hat_of, 'pR2': float(np.nanmean(pr2_of))}

    print(f'  nfilters={nf:2d}  VR pR²={results_nf_vr[nf]["pR2"]:+.4f}  '
          f'OF1 pR²={results_nf_of[nf]["pR2"]:+.4f}')

print('\nDone.')

In [ ]:
fig_nf_pr2, ax = plt.subplots(figsize=(7, 3.5))

pr2_vr_nf = [results_nf_vr[nf]['pR2'] for nf in NFILTERS_SWEEP]
pr2_of_nf = [results_nf_of[nf]['pR2'] for nf in NFILTERS_SWEEP]

ax.plot(NFILTERS_SWEEP, pr2_vr_nf, 'o-', color=COL_GC,    lw=2, ms=6, label='VR')
ax.plot(NFILTERS_SWEEP, pr2_of_nf, 's-', color='#3171ae',  lw=2, ms=6, label='OF1')

# Mark the FIXED_NFILTERS=5 used in the history sweep
ax.axvline(FIXED_NFILTERS, color='#888888', lw=1.2, ls='--', alpha=0.8,
           label=f'Fixed n_filters = {FIXED_NFILTERS} (used in history sweep)')

# Annotate peak if visible
best_vr = NFILTERS_SWEEP[np.argmax(pr2_vr_nf)]
best_of = NFILTERS_SWEEP[np.argmax(pr2_of_nf)]
ax.scatter([best_vr], [max(pr2_vr_nf)], color=COL_GC,   s=100, zorder=5,
           marker='*', label=f'VR peak at n={best_vr}')
ax.scatter([best_of], [max(pr2_of_nf)], color='#3171ae', s=100, zorder=5,
           marker='*', label=f'OF1 peak at n={best_of}')

ax.set_xlabel('n_filters', fontsize=10)
ax.set_ylabel('pR²', fontsize=10)
ax.set_title(
    f'pR² vs n_filters  (history = {HISTORY_NF} ms, GC {TARGET_ID}, '
    f'{len(cov_gc_ids)} GC covariates)',
    fontsize=9
)
ax.set_xticks(NFILTERS_SWEEP)
ax.axhline(0, color='#cccccc', lw=0.8, ls='--')
ax.legend(fontsize=8, frameon=False)
ax.spines[['top', 'right']].set_visible(False)
ax.tick_params(labelsize=8)
plt.tight_layout()
plt.savefig(fig_path + f'nfilters_pr2_M{mouse}D{day}_GC{TARGET_ID}.pdf',
            bbox_inches='tight', dpi=150)
plt.show()
print(f'VR  peak: nfilters={best_vr}  pR²={max(pr2_vr_nf):+.4f}')
print(f'OF1 peak: nfilters={best_of}  pR²={max(pr2_of_nf):+.4f}')

In [ ]:
# ── Reconstruct rate maps for NFILTERS_SHOW = [1, 5, 10, 20] ────────────────
_excl_nf   = {TARGET_ID} | set(cov_gc_ids)
_pseudo_nf = [c for c in all_cells.cluster_id.values.astype(int)
               if c not in _excl_nf]
assert len(_pseudo_nf) >= len(NFILTERS_SHOW), 'Not enough pseudo IDs'

# ── VR reconstruction ────────────────────────────────────────────────────────
exp_spk_nf_vr = {_pseudo_nf[j]: results_nf_vr[nf]['Y_hat']
                  for j, nf in enumerate(NFILTERS_SHOW)}
print('Reconstructing VR rate maps...')
tcs_recon_nf_vr, _, _, lb_nf_vr, _, _ = compute_vr_tcs_using_expected_spikes(
    mouse, day, apply_zscore=False, vr_type='VR',
    source_path=source_path, expected_spikes=exp_spk_nf_vr)

rm_pred_nf_vr = {
    nf: gaussian_filter(
        np.nan_to_num(tcs_recon_nf_vr[_pseudo_nf[j]]).astype(np.float64), sigma=SIGMA
    )[:lb_nf_vr]
    for j, nf in enumerate(NFILTERS_SHOW)
}

# ── OF1 reconstruction ───────────────────────────────────────────────────────
exp_spk_nf_of = {_pseudo_nf[j]: results_nf_of[nf]['Y_hat']
                  for j, nf in enumerate(NFILTERS_SHOW)}
print('Reconstructing OF1 rate maps...')
tcs_recon_nf_of, _, _, _, _ = compute_of_tcs_using_expected_spikes(
    mouse, day, apply_zscore=False, apply_guassian_filter=True,
    source_path=source_path, expected_spikes=exp_spk_nf_of,
    optimal_shift=optimal_shift)

rm_pred_nf_of = {}
for j, nf in enumerate(NFILTERS_SHOW):
    raw = np.nan_to_num(tcs_recon_nf_of[_pseudo_nf[j]]).astype(np.float64)
    raw[raw == 0] = np.nan
    filled = _fill_nans_from_neighbors(raw)
    rm_pred_nf_of[nf] = gaussian_filter(np.nan_to_num(filled), sigma=SIGMA)

print('Computing autocorrelograms...')
ac_pred_nf_of = {nf: autocorr2d(rm_pred_nf_of[nf]) for nf in NFILTERS_SHOW}
print('Done.')

In [ ]:
# ── Combined figure: filters | pR² line | VR maps | OF1 maps | autocorrs ─────
n_show = len(NFILTERS_SHOW)
n_cols = n_show + 1   # +1 for True

fig_nf = plt.figure(figsize=(n_cols * 2.2, 12))
gs_nf  = gridspec.GridSpec(
    4, n_cols, figure=fig_nf,
    height_ratios=[1.0, 2.8, 1.8, 1.8],
    hspace=0.40, wspace=0.08,
    left=0.07, right=0.97, top=0.95, bottom=0.05
)

CMAP_VR = white_to_hex_cmap(COL_GC)
CMAP_OF = 'viridis'
CMAP_AC = 'RdBu_r'
filt_cols = plt.cm.viridis(np.linspace(0.15, 0.9, max(NFILTERS_SHOW)))

# ── Row 0: filter bank overlays ───────────────────────────────────────────────
# Col 0: empty / label
ax0 = fig_nf.add_subplot(gs_nf[0, 0])
ax0.axis('off')
ax0.text(0.5, 0.5, 'Filter\nbanks', ha='center', va='center',
         fontsize=9, fontweight='bold', transform=ax0.transAxes)

for ci, nf in enumerate(NFILTERS_SHOW):
    ax = fig_nf.add_subplot(gs_nf[0, ci + 1])
    t_ms, filts = get_filters(HISTORY_NF, nf)
    cols_nf = plt.cm.viridis(np.linspace(0.15, 0.9, nf))
    for k, (f, c) in enumerate(zip(filts, cols_nf)):
        ax.plot(t_ms, f, color=c, lw=1.5, alpha=0.8)
        peak_t = t_ms[np.argmax(f)]
        ax.axvline(peak_t, color=c, lw=0.6, ls='--', alpha=0.4)
    ax.set_title(f'n_filters = {nf}', fontsize=8, fontweight='bold')
    ax.set_xlim(0, HISTORY_NF)
    ax.set_ylim(bottom=0)
    ax.set_xlabel('Lag (ms)', fontsize=7)
    ax.tick_params(labelsize=6)
    ax.spines[['top', 'right']].set_visible(False)
    if ci > 0: ax.set_yticklabels([])
    else: ax.set_ylabel('Weight', fontsize=7)

# ── Row 1: VR rate maps ───────────────────────────────────────────────────────
ax = fig_nf.add_subplot(gs_nf[1, 0])
plot_firing_rate_map(ax, rm_true_vr, bs=bs, tl=tl, p=95, cmap=CMAP_VR)
ax.set_title('True', fontsize=8, fontweight='bold')
ax.set_ylabel('VR  Trial', fontsize=8)
ax.tick_params(labelsize=7)

for ci, nf in enumerate(NFILTERS_SHOW):
    ax = fig_nf.add_subplot(gs_nf[1, ci + 1])
    plot_firing_rate_map(ax, rm_pred_nf_vr[nf], bs=bs, tl=tl, p=95, cmap=CMAP_VR)
    ax.set_title(f'pR²={results_nf_vr[nf]["pR2"]:+.3f}', fontsize=8, color=COL_GC)
    ax.set_yticks([]); ax.tick_params(labelsize=7)

# ── Row 2: OF1 rate maps ──────────────────────────────────────────────────────
vmax_of_nf = max(np.nanmax(rm_true_of),
                 max(np.nanmax(rm_pred_nf_of[nf]) for nf in NFILTERS_SHOW))

ax = fig_nf.add_subplot(gs_nf[2, 0])
ax.imshow(rm_true_of, origin='lower', cmap=CMAP_OF,
          vmin=0, vmax=vmax_of_nf, interpolation='nearest')
ax.set_title('True', fontsize=8, fontweight='bold')
ax.set_ylabel('OF1', fontsize=8); ax.set_xticks([]); ax.set_yticks([])

for ci, nf in enumerate(NFILTERS_SHOW):
    ax = fig_nf.add_subplot(gs_nf[2, ci + 1])
    ax.imshow(rm_pred_nf_of[nf], origin='lower', cmap=CMAP_OF,
              vmin=0, vmax=vmax_of_nf, interpolation='nearest')
    ax.set_title(f'pR²={results_nf_of[nf]["pR2"]:+.3f}', fontsize=8, color='#3171ae')
    ax.set_xticks([]); ax.set_yticks([])

# ── Row 3: autocorrelograms ───────────────────────────────────────────────────
vmax_ac_nf = np.nanpercentile(
    np.abs(ac_true_of[~np.isnan(ac_true_of)]), 95)

ax = fig_nf.add_subplot(gs_nf[3, 0])
ax.imshow(ac_true_of, origin='lower', cmap=CMAP_AC,
          vmin=-vmax_ac_nf, vmax=vmax_ac_nf, interpolation='nearest')
ax.set_title('True', fontsize=8, fontweight='bold')
ax.set_ylabel('Autocorr', fontsize=8); ax.set_xticks([]); ax.set_yticks([])

for ci, nf in enumerate(NFILTERS_SHOW):
    ax = fig_nf.add_subplot(gs_nf[3, ci + 1])
    ax.imshow(ac_pred_nf_of[nf], origin='lower', cmap=CMAP_AC,
              vmin=-vmax_ac_nf, vmax=vmax_ac_nf, interpolation='nearest')
    ax.set_xticks([]); ax.set_yticks([])

fig_nf.suptitle(
    f'n_filters sweep — history = {HISTORY_NF} ms  |  '
    f'GC {TARGET_ID},  {len(cov_gc_ids)} GC covariates',
    fontsize=10, fontweight='bold'
)

save_nf = fig_path + f'nfilters_rate_maps_M{mouse}D{day}_GC{TARGET_ID}.pdf'
fig_nf.savefig(save_nf, bbox_inches='tight', dpi=300)
plt.show()
print(f'Saved → {save_nf}')